# Chạy thực nghiệm trên Google Colab

**Trước khi bắt đầu:** Runtime → Change runtime type → **GPU (T4)**.

Chạy lần lượt từng cell theo thứ tự. Code được clone từ GitHub; dữ liệu CICIDS2017 đọc từ Google Drive.

**Thứ tự bắt buộc:** Bước 5 → 6 → 7. Bước 7 (`populate_latex_tables.py`) phải chạy **cuối cùng** thì 5 bảng trong bài mới được điền số.

## Bước 1 — Clone code từ GitHub
Xóa bản cũ (nếu có) rồi clone lại để chắc chắn lấy code mới nhất.

In [ ]:
%cd /content
!rm -rf IEEECS_CPS_2026_paper
!git clone https://github.com/vuthainguyen1602/IEEECS_CPS_2026_paper.git
# Phải thấy: code  data ...  (nếu 'repository not found' -> kiểm tra URL repo bằng `git remote -v` trên máy)
!ls IEEECS_CPS_2026_paper/paper_1

## Bước 2 — Gắn Google Drive
Dữ liệu CICIDS2017 (8 file CSV) nằm trên Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 3 — Đặt đường dẫn và kiểm tra dữ liệu
Sửa `RAW_DATA_DIR` cho đúng thư mục chứa 8 file CSV trên Drive của bạn.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR  = "/content/IEEECS_CPS_2026_paper/paper_1"
RAW_DATA_DIR = "/content/drive/MyDrive/ids-2017"   # <-- SUA cho dung thu muc chua 8 CSV

assert Path(PROJECT_DIR).exists(), "Chua thay PROJECT_DIR — chay lai Buoc 1 (clone)."
os.chdir(PROJECT_DIR)
n = len(list(Path(RAW_DATA_DIR).glob("*.csv"))) + len(list(Path(RAW_DATA_DIR).glob("*.CSV")))
print("Thu muc lam viec:", os.getcwd())
print("So file CSV tim thay:", n)
assert n >= 8, "Chua thay du 8 CSV — sua lai RAW_DATA_DIR (thu them /MachineLearningCVE)."

## Bước 4 — Cài thư viện

In [ ]:
!pip install -r requirements.txt

## Bước 5 — Chuẩn bị dữ liệu
Đọc CSV từ Drive, làm sạch, tạo nhãn nhị phân, chia train/test và lưu Parquet.

In [ ]:
!python code/data_preparation.py --input-dir "{RAW_DATA_DIR}"

## Bước 6 — Huấn luyện Teacher rồi Chưng cất tri thức
Kiểm tra log: phải thấy `[Ablation 1/4]…4/4`, dòng `Minority class = …%`, và KD in **4 dòng** Teacher / Student-Soft / Student-Hard / Student-Direct.

In [ ]:
!python code/hybrid_ids_cicids2017.py
!python code/knowledge_distillation.py

## Bước 7 — Vẽ lại hình và tự điền 5 bảng (CHẠY CUỐI CÙNG)
`populate_latex_tables.py` **không được** in dòng `[warn] pattern not found`.

In [ ]:
!python code/replot_figures.py
!python code/populate_latex_tables.py

## Bước 8 — Đóng gói kết quả và tải về
Tải về `sections/` (đã điền số), `figures/`, `references.bib` và 2 file JSON kết quả.

In [ ]:
%cd /content/IEEECS_CPS_2026_paper
!zip -r /content/output.zip sections figures references.bib paper_1/results/all_results.json paper_1/results/kd_results.json
from google.colab import files
files.download('/content/output.zip')

---
Sau khi tải về: giải nén, copy đè `sections/`, `figures/`, `references.bib` vào dự án trên máy, rồi build:

```bash
latexmk -C VNICT2026_Template_LaTeX.tex
latexmk -xelatex -bibtex VNICT2026_Template_LaTeX.tex
```